# Multi-fidelity candidate progression

This notebook evaluates whether a controlled **lab → glasshouse → field** progression policy can reduce expensive field-selection regret under a fixed experimental budget.

The public fungicide case in this repository does not contain harmonised measurements at all three fidelities for the same candidates, so this experiment is intentionally synthetic with known latent field truth. Historical calibration is nevertheless trained only against noisy observed field means.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from crop_protection_ps.multifidelity_demo import run_multifidelity_demo
from crop_protection_ps.multifidelity import MultiFidelityConfig

config = MultiFidelityConfig()
config.common_budget

1040

## Locked cost structure

One replicate costs 1 relative unit in the lab, 4 in the glasshouse and 20 in the field. Each observed candidate receives two replicates at any stage it reaches. The stage counts are chosen so all policies spend exactly the same total budget.

In [2]:
summary = run_multifidelity_demo(ROOT)
summary["fidelity_costs"]

{'lab': 1, 'glasshouse': 4, 'field': 20, 'common_budget_units': 1040}

## Cross-fidelity calibration gate

The glasshouse stage is not admitted merely because it is cheaper than field work. It must first improve held-out prediction of latent field efficacy relative to lab evidence alone. The models are trained against **observed historical field means**, never the simulated latent truth.

In [3]:
calibration = pd.read_csv(ROOT / "results" / "multifidelity" / "calibration_metrics.csv")
calibration

,calibration_model,rmse_field_efficacy,r2_field_efficacy,correlation
0,lab_only,0.732499,0.397419,0.636146
1,lab_plus_glasshouse,0.470048,0.751866,0.867933


The pre-specified gate requires at least a 10% held-out RMSE reduction. In the locked DGP, lab + glasshouse evidence improves RMSE by about 35.8%, so the glasshouse stage earns promotion.

In [4]:
costs = pd.read_csv(ROOT / "results" / "multifidelity" / "policy_costs.csv")
costs

,policy,lab_candidates,glasshouse_candidates,field_candidates,total_cost_units
0,field_only,0,0,26,1040
1,lab_field,80,0,22,1040
2,multifidelity,80,30,16,1040


## Equal-budget progression decision

Each rollout contains 80 new candidates. The final decision selects eight. The oracle is the eight candidates with highest latent field efficacy. We report oracle recall and simple regret,

\[
R = ar	heta_{oracle} - ar	heta_{selected}.
\]

The policies share candidate truth and observations within each rollout, making policy differences paired Monte Carlo comparisons.

In [5]:
metrics = pd.read_csv(ROOT / "results" / "multifidelity" / "screening_policy_metrics.csv")
metrics

,policy,n_rollouts,mean_oracle_top_k_recall,mcse_oracle_top_k_recall,mean_simple_regret_field_efficacy,mcse_simple_regret_field_efficacy,mean_selected_mean_true_field_efficacy,mcse_selected_mean_true_field_efficacy
0,field_only,5000,0.324575,0.002210,0.663261,0.003586,3.052942,0.003939
1,lab_field,5000,0.648725,0.002035,0.202020,0.001773,3.514183,0.003614
2,multifidelity,5000,0.707625,0.001861,0.140080,0.001331,3.576123,0.003466


In [6]:
result = summary["equal_budget_policy_result"]
{
    "multifidelity_regret_reduction_vs_lab_field_percent": result["multifidelity_regret_reduction_vs_lab_field_percent"],
    "paired_regret_difference": result["paired_regret_difference_multifidelity_minus_lab_field"],
    "paired_recall_difference": result["paired_recall_difference_multifidelity_minus_lab_field"],
}

{'multifidelity_regret_reduction_vs_lab_field_percent': 30.66048497738,
 'paired_regret_difference': {'mean_difference': -0.06194030898095661,
  'mc_standard_error': 0.0013049434467904567,
  'mc95_low': -0.0644979981366659,
  'mc95_high': -0.05938261982524731},
 'paired_recall_difference': {'mean_difference': 0.0589,
  'mc_standard_error': 0.0014414657695252225,
  'mc95_low': 0.056074727091730565,
  'mc95_high': 0.061725272908269437}}

## Interpretation

The result is not that every lower-fidelity assay should drive field progression. It is that a lower-fidelity stage becomes decision-relevant only after its **transfer to field performance** has survived held-out validation and after the resulting progression policy is shown to improve the final decision under equal cost.

The numerical gains are properties of this explicit DGP. They are not empirical claims about any commercial Crop Protection pipeline.